# KKBox Hybrid Retrieval Baseline + SSL (CL4SRec-style)

This notebook is a **clean baseline pipeline** for AML/IRRT courses projects using the **KKBox Music Recommendation Challenge** dataset.

**Pipeline overview**
1. **Download** KKBox Dataset.  
2. **Load & clean** `train.csv`, `songs.csv`, `members.csv`, `song_extra_info.csv`.  
3. **Feature extraction** (lightweight):
   - Interaction context: `source_system_tab`, `source_screen_name`, `source_type`
   - Item metadata: `language`, `genre_main`, `len_bucket`, `artist_bucket`
   - Metadata document construction for sparse retrieval (text fields concatenation for BM25 indexing)
4. **Sequence construction**: per-user ordered sequences.
   - This challenge dataset has no explicit event timestamps; we use **file order** as a proxy time (baseline).
5. **Models to compare**
   - **TopPop** (popularity)
   - **SASRec** (supervised next-item)
   - **SASRec + SSL** (CL4SRec-style InfoNCE pretraining)
   - Optional **debias knob**: popularity-weighted InfoNCE (DCRec-inspired, baseline-only)
6. **Retrieval components**
   - **Sparse retrieval (BM25)** over item metadata documents
   - **Dense retrieval (Faiss)** over learned item embeddings (exact or ANN index)
   - **Hybrid retrieval**: union of dense and sparse candidates followed by fusion and reranking
7. **Hybrid evaluation**
   - Dense-only retrieval
   - Sparse-only retrieval
   - Hybrid (Dense ∪ Sparse → Fusion → Rerank)

**Metrics**
- Recall@K, NDCG@K, MRR  
- MAP@K (for retrieval completeness)  
- Latency breakdown (dense retrieval, sparse retrieval, fusion/rerank, total per-user time)

## 0) Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np

import os
from pathlib import Path
import shutil
import zipfile
import gdown

from typing import Dict, List
from collections import Counter

from dataclasses import dataclass

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
import time

import re
import unicodedata

from rank_bm25 import BM25Okapi

In [35]:
PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "data"
KKBOX_DIR = DATA_DIR / "kkbox"
KKBOX_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED = ["train.csv", "songs.csv", "members.csv", "song_extra_info.csv"]

# Subsampling / speed knobs (safe defaults)
MAX_USERS = int(os.environ.get("MAX_USERS", "200000"))  # keep top-N most active users
MIN_USER_INTERACTIONS = int(os.environ.get("MIN_USER_INTERACTIONS", "10"))
MAX_ITEMS = int(os.environ.get("MAX_ITEMS", "50000"))  # cap item vocab for speed
MAX_SEQ_LEN = int(os.environ.get("MAX_SEQ_LEN", "100"))

# Training knobs
D_MODEL = int(os.environ.get("D_MODEL", "128"))
N_HEADS = int(os.environ.get("N_HEADS", "4"))
N_LAYERS = int(os.environ.get("N_LAYERS", "2"))
DROPOUT = float(os.environ.get("DROPOUT", "0.1"))

BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "512"))
LR = float(os.environ.get("LR", "1e-3"))
EPOCHS = int(os.environ.get("EPOCHS", "50"))

# SSL knobs
DO_SSL = os.environ.get("DO_SSL", "1") == "1"
SSL_EPOCHS = int(os.environ.get("SSL_EPOCHS", "50"))
SSL_TAU = float(os.environ.get("SSL_TAU", "0.2"))

# Debias knob (DCRec-inspired): popularity weighting inside InfoNCE
DEBIAS_INFO_NCE = os.environ.get("DEBIAS_INFO_NCE", "1") == "1"


# Retrieval eval knobs
RETR_K = int(os.environ.get("RETR_K", "200"))  # candidates per head
FUSE_MAX = int(os.environ.get("FUSE_MAX", "500"))  # fused candidate budget
FINAL_K = int(os.environ.get("FINAL_K", "20"))  # report @K

EVAL_USERS = int(os.environ.get("EVAL_USERS", "5000"))
EVAL_BATCH = int(os.environ.get("EVAL_BATCH", "256"))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("KKBOX_DIR:", KKBOX_DIR)
print("Filter:", dict(MAX_USERS=MAX_USERS, MIN_USER_INTERACTIONS=MIN_USER_INTERACTIONS, MAX_ITEMS=MAX_ITEMS, MAX_SEQ_LEN=MAX_SEQ_LEN))
print("Model:", dict(D_MODEL=D_MODEL, N_HEADS=N_HEADS, N_LAYERS=N_LAYERS, DROPOUT=DROPOUT))
print("Train:", dict(BATCH_SIZE=BATCH_SIZE, LR=LR, EPOCHS=EPOCHS))
print("SSL:", dict(DO_SSL=DO_SSL, SSL_EPOCHS=SSL_EPOCHS, SSL_TAU=SSL_TAU, DEBIAS_INFO_NCE=DEBIAS_INFO_NCE))
print("Eval:", dict(RETR_K=RETR_K, FUSE_MAX=FUSE_MAX, FINAL_K=FINAL_K, EVAL_USERS=EVAL_USERS, EVAL_BATCH=EVAL_BATCH))
print("DEVICE:", DEVICE)


KKBOX_DIR: C:\Users\sokos\DataspellProjects\MML_Project\data\kkbox
Filter: {'MAX_USERS': 200000, 'MIN_USER_INTERACTIONS': 10, 'MAX_ITEMS': 50000, 'MAX_SEQ_LEN': 100}
Model: {'D_MODEL': 128, 'N_HEADS': 4, 'N_LAYERS': 2, 'DROPOUT': 0.1}
Train: {'BATCH_SIZE': 512, 'LR': 0.001, 'EPOCHS': 50}
SSL: {'DO_SSL': True, 'SSL_EPOCHS': 50, 'SSL_TAU': 0.2, 'DEBIAS_INFO_NCE': True}
Eval: {'RETR_K': 200, 'FUSE_MAX': 500, 'FINAL_K': 20, 'EVAL_USERS': 5000, 'EVAL_BATCH': 256}
DEVICE: cuda


## 1) Download KKBox Dataset

In [36]:
def has_all_required_files(folder: Path) -> bool:
    return all((folder / f).exists() for f in REQUIRED)

def remove_possible_previous_extracts(base: Path):
    """
    Remove previous extracted files/dirs to ensure a clean overwrite.
    We delete:
      - required csv files in base
      - common nested duplicate folder base/kkbox
    """
    # delete required files if exist
    for f in REQUIRED:
        p = base / f
        if p.exists():
            p.unlink()

    # delete nested duplicate folder if exists (data/kkbox/kkbox)
    nested = base / "kkbox"
    if nested.exists() and nested.is_dir():
        shutil.rmtree(nested)

# 1) If everything already present in KKBOX_DIR, do nothing.
if has_all_required_files(KKBOX_DIR):
    print(f"KKBox files already exist in {KKBOX_DIR} — skipping download.")
else:
    print("Missing required KKBox files — re-downloading and fully overwriting destination.")

    # 2) Clean destination (only what we need to overwrite)
    remove_possible_previous_extracts(KKBOX_DIR)

    # 3) Download zip
    file_id = "1SYEYKpb67ENakrz-gYlqJgyrGsuPGc8y"
    url = f"https://drive.google.com/uc?id={file_id}"
    output_zip = KKBOX_DIR / "kkbox.zip"

    if output_zip.exists():
        output_zip.unlink()

    gdown.download(url, str(output_zip), quiet=False)

    # 4) Extract
    with zipfile.ZipFile(output_zip, "r") as z:
        z.extractall(KKBOX_DIR)

    # 5) Fix duplicate folder (data/kkbox/kkbox -> data/kkbox)
    nested = KKBOX_DIR / "kkbox"
    if nested.exists() and nested.is_dir():
        # move everything up, overwriting if needed
        for src in nested.iterdir():
            dst = KKBOX_DIR / src.name
            if dst.exists():
                if dst.is_dir():
                    shutil.rmtree(dst)
                else:
                    dst.unlink()
            shutil.move(str(src), str(dst))
        shutil.rmtree(nested)

    # 6) Remove zip after successful extraction
    output_zip.unlink(missing_ok=True)

    # 7) Final check
    if not has_all_required_files(KKBOX_DIR):
        missing = [f for f in REQUIRED if not (KKBOX_DIR / f).exists()]
        raise FileNotFoundError(f"After extraction, still missing required files: {missing} in {KKBOX_DIR}")

    print(f"Done. KKBox dataset is ready in: {KKBOX_DIR}")

KKBox files already exist in C:\Users\sokos\DataspellProjects\MML_Project\data\kkbox — skipping download.


## 2) Load and clean KKBox tables

In [37]:
train = pd.read_csv(KKBOX_DIR / "train.csv")
songs = pd.read_csv(KKBOX_DIR / "songs.csv")
members = pd.read_csv(KKBOX_DIR / "members.csv")
extra = pd.read_csv(KKBOX_DIR / "song_extra_info.csv")

print("train:", train.shape, "songs:", songs.shape, "members:", members.shape, "extra:", extra.shape)
train.head()

train: (7377418, 6) songs: (2296320, 7) members: (34403, 7) extra: (2295971, 3)


,msno,song_id,source_system_tab,source_screen_name,source_type,target
0,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,BBzumQNXUHKdEBOB7mAJuzok+IJA1c2Ryg/yzTF6tik=,explore,Explore,online-playlist,1
1,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,bhp/MpSNoqoxOIB+/l8WPqu6jldth4DIpCm3ayXnJqM=,my library,Local playlist more,local-playlist,1
2,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,JNWfrrC7zNN7BdMpsISKa4Mw+xVJYNnxXh3/Epw7QgY=,my library,Local playlist more,local-playlist,1
3,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,2A87tzfnJTSWqD7gIZHisolhe4DMdzkbd6LzO1KHjNs=,my library,Local playlist more,local-playlist,1
4,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,3qm6XTZ6MOCU11x8FIVbAGH5l5uMkT3/ZalWG1oo2Gc=,explore,Explore,online-playlist,1


In [38]:
# 2.1 Clean interactions: use target==1 as implicit positives
KEEP_ONLY_POSITIVES = True

if KEEP_ONLY_POSITIVES and "target" in train.columns:
    train = train[train["target"] == 1].copy()

keep_cols = ["msno", "song_id", "source_system_tab", "source_screen_name", "source_type"]
keep_cols = [c for c in keep_cols if c in train.columns]
train = train[keep_cols].copy()

for c in ["source_system_tab", "source_screen_name", "source_type"]:
    if c in train.columns:
        train[c] = train[c].fillna("UNK").astype(str)

train["msno"] = train["msno"].astype(str)
train["song_id"] = train["song_id"].astype(str)

train = train.drop_duplicates().reset_index(drop=True)

# No timestamps in KKBox dataset -> proxy time by row order
train["ts"] = np.arange(len(train), dtype=np.int64)

print("Cleaned interactions:", train.shape)
train.head()

Cleaned interactions: (3714656, 6)


,msno,song_id,source_system_tab,source_screen_name,source_type,ts
0,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,BBzumQNXUHKdEBOB7mAJuzok+IJA1c2Ryg/yzTF6tik=,explore,Explore,online-playlist,0
1,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,bhp/MpSNoqoxOIB+/l8WPqu6jldth4DIpCm3ayXnJqM=,my library,Local playlist more,local-playlist,1
2,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,JNWfrrC7zNN7BdMpsISKa4Mw+xVJYNnxXh3/Epw7QgY=,my library,Local playlist more,local-playlist,2
3,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,2A87tzfnJTSWqD7gIZHisolhe4DMdzkbd6LzO1KHjNs=,my library,Local playlist more,local-playlist,3
4,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,3qm6XTZ6MOCU11x8FIVbAGH5l5uMkT3/ZalWG1oo2Gc=,explore,Explore,online-playlist,4


In [ ]:
# 2.2 Clean item metadata + extract lightweight features
UNK = "UNK"
LANG_FALLBACK = -1
ARTIST_HASH_MOD = 50_000

DOC_COLS = ["name", "artist_name", "composer", "lyricist", "genre_ids", "isrc"]

songs = songs.copy()
songs["song_id"] = songs["song_id"].astype(str)

songs["language"] = pd.to_numeric(songs.get("language", LANG_FALLBACK), errors="coerce").fillna(LANG_FALLBACK).astype(int)
songs["artist_name"] = songs.get("artist_name", UNK).fillna(UNK).astype(str)
songs["genre_ids"] = songs.get("genre_ids", UNK).fillna(UNK).astype(str)
songs["song_length"] = pd.to_numeric(songs.get("song_length", -1), errors="coerce").fillna(-1).astype(int)

genre_split = songs["genre_ids"].str.split("|", n=1, expand=True)
songs["genre_main"] = np.where(songs["genre_ids"].eq(UNK), UNK, genre_split[0].fillna(UNK)).astype(str)

def length_bucket_ms(ms: int) -> str:
    if ms <= 0:
        return UNK
    s = ms / 1000.0
    if s < 60:
        return "<1m"
    if s < 120:
        return "1-2m"
    if s < 180:
        return "2-3m"
    if s < 240:
        return "3-4m"
    if s < 300:
        return "4-5m"
    return ">=5m"

songs["len_bucket"] = songs["song_length"].map(length_bucket_ms).astype(str)

def artist_bucket(name: str, mod: int = ARTIST_HASH_MOD) -> str:
    return f"a_{abs(hash(name)) % mod}"

songs["artist_bucket"] = songs["artist_name"].map(artist_bucket).astype(str)

songs_feats = songs[["song_id", "language", "genre_main", "len_bucket", "artist_bucket"]].copy()
display(songs_feats.head())

extra = extra.copy()
extra["song_id"] = extra["song_id"].astype(str)

songs_full = songs.merge(extra, on="song_id", how="left")

for c in DOC_COLS:
    if c not in songs_full.columns:
        songs_full[c] = ""

for c in DOC_COLS:
    songs_full[c] = songs_full[c].fillna("").astype(str)

songs_full["language"] = pd.to_numeric(songs_full.get("language", LANG_FALLBACK), errors="coerce").fillna(LANG_FALLBACK).astype(int)

text_block = songs_full[DOC_COLS].agg(" ".join, axis=1).str.strip()
text_block = text_block.str.replace(r"\s+", " ", regex=True)

songs_full["doc"] = (text_block + " " + songs_full["language"].map(lambda x: f"lang_{int(x)}")).str.strip().str.lower()

songid_to_doc = dict(zip(songs_full["song_id"], songs_full["doc"]))

display(songs_full[["song_id", "doc"]].head())

,song_id,doc
0,CXoTN1eb7AI+DntdU1vbcwGRV4SCIDxZu+YD8JP8r4E=,焚情 張信哲 (jeff chang) 董貞 何啟弘 465 twb531410010 la...
1,o0kFgae9QtnYgRkVPqLJwa05zIhRlUjfF7O1tDw0ZDU=,playing with fire blackpink teddy| future bou...
2,DwVvVurfpuz+XPuFvucclVQEyPqcpUkHR0ne1RQzPs0=,sorry| sorry super junior 465 lang_31
3,dKMBWoZyScdxSkihKG+Vf47nc18N9q4m58+b4e7dSSE=,愛我的資格 s.h.e 湯小康 徐世珍 465 twc950206108 lang_3
4,W3bqWd3T+VeHFzHAUfARgW9AvVRaF4N5Yzm4Mr6Eo/o=,mary had a little lamb 貴族精選 traditional tradit...


In [40]:
# 2.3 User metadata
members = members.copy()
members["msno"] = members["msno"].astype(str)
for c in ["city", "bd", "gender", "registered_via"]:
    if c in members.columns:
        members[c] = members[c].fillna("UNK").astype(str)

members.head()

,msno,city,bd,gender,registered_via,registration_init_time,expiration_date
0,XQxgAYj3klVKjR3oxPPXYYFp4soD4TuBghkhMTD4oTw=,1,0,UNK,7,20110820,20170920
1,UizsfmJb9mV54qE9hCYyU07Va97c0lCRLEQX3ae+ztM=,1,0,UNK,7,20150628,20170622
2,D8nEhsIOBSoE6VthTaqDX8U6lqjJ7dLdr72mOyLya2A=,1,0,UNK,4,20160411,20170712
3,mCuD+tZ1hERA/o5GPqk38e041J8ZsBaLcu7nGoIIvhI=,1,0,UNK,9,20150906,20150907
4,q4HRBfVSssAFS9iRfxWrohxuk9kCYMKjHOEagUMV6rQ=,1,0,UNK,4,20170126,20170613


## 3) Build interaction table + remap IDs

We join item metadata into interactions, then:
- filter users by minimum interactions
- keep top-N users/items for speed
- remap `user_id` and `item_id` to contiguous integers

In [41]:
def remap_series_to_int(s: pd.Series):
    uniq = s.unique()
    mapping = {v: i + 1 for i, v in enumerate(uniq)}
    return s.map(mapping).astype(np.int32), mapping

def filter_users_by_min_interactions(df: pd.DataFrame, min_k: int):
    cnt = df.groupby("msno").size()
    good = cnt[cnt >= min_k].index
    return df[df["msno"].isin(good)].copy()

def cap_top_users(df: pd.DataFrame, max_users: int):
    if df["msno"].nunique() <= max_users:
        return df
    top_users = df.groupby("msno").size().sort_values(ascending=False).head(max_users).index
    return df[df["msno"].isin(top_users)].copy()

def cap_top_items(df: pd.DataFrame, max_items: int):
    if df["song_id"].nunique() <= max_items:
        return df
    top_items = df.groupby("song_id").size().sort_values(ascending=False).head(max_items).index
    return df[df["song_id"].isin(top_items)].copy()

df = train.merge(songs_feats, on="song_id", how="left")

for c in ["language", "genre_main", "len_bucket", "artist_bucket"]:
    df[c] = df[c].fillna("UNK")

df = filter_users_by_min_interactions(df, MIN_USER_INTERACTIONS)
df = cap_top_users(df, MAX_USERS)
df = cap_top_items(df, MAX_ITEMS)

df = df.sort_values(["msno", "ts"]).reset_index(drop=True)

df["user_id"], user_map = remap_series_to_int(df["msno"])
df["item_id"], item_map = remap_series_to_int(df["song_id"])

n_users = int(df["user_id"].max())
n_items = int(df["item_id"].max())

print("Filtered interactions:", df.shape, "| users:", n_users, "| items:", n_items)
df.head()

Filtered interactions: (3403634, 12) | users: 22072 | items: 50000


,msno,song_id,source_system_tab,source_screen_name,source_type,ts,language,genre_main,len_bucket,artist_bucket,user_id,item_id
0,++5wYjoMgQHoRuD3GbbvmphZbBBwymzv5Q4l8sywtuU=,aRNy4rKgWyDwTjsMIcAbtpGvddDZ7L7OGUeJ09LAM2w=,discover,UNK,top-hits-for-artist,458279,52.0,1616,3-4m,a_1996,1,1
1,++5wYjoMgQHoRuD3GbbvmphZbBBwymzv5Q4l8sywtuU=,J4qKkLIoW7aYACuTupHLAPZYmRp08en1AEux+GSUzdw=,discover,UNK,top-hits-for-artist,458280,52.0,1616,3-4m,a_1996,1,2
2,++5wYjoMgQHoRuD3GbbvmphZbBBwymzv5Q4l8sywtuU=,zHqZ07gn+YvF36FWzv9+y8KiCMhYhdAUS+vSIKY3UZY=,discover,UNK,top-hits-for-artist,458281,52.0,1616,3-4m,a_1996,1,3
3,++5wYjoMgQHoRuD3GbbvmphZbBBwymzv5Q4l8sywtuU=,v/3onppBGoSpGsWb8iaCIO8eX5+iacbH5a4ZUhT7N54=,discover,UNK,top-hits-for-artist,458282,52.0,1616,2-3m,a_1996,1,4
4,++5wYjoMgQHoRuD3GbbvmphZbBBwymzv5Q4l8sywtuU=,upH6pOAUd+iV/MkpzeELvqEFoTEIVhsV9eML8N7/gUM=,my library,Local playlist more,local-library,555126,52.0,2022,3-4m,a_35429,1,5


In [42]:
# Sanity check: item frequency distribution + unique items
item_freq = df["item_id"].value_counts()
print("Most frequent item_id:", int(item_freq.index[0]), "count:", int(item_freq.iloc[0]))
print("Unique items:", df["item_id"].nunique())

Most frequent item_id: 353 count: 10476
Unique items: 50000


## 4) Sequences + split

Create dataset splits per user:
- train = seq[:-2]
- valid = seq[:-1]
- test  = seq[:]

We also compute **TopPop** baseline items from train sequences

In [43]:
def left_pad(seq: List[int], max_len: int, pad: int = 0) -> List[int]:
    if len(seq) >= max_len:
        return seq[-max_len:]
    return [pad] * (max_len - len(seq)) + seq

@dataclass
class SeqData:
    user_seqs: Dict[int, List[int]]
    n_users: int
    n_items: int

user_seqs = df.groupby("user_id")["item_id"].apply(list).to_dict()
seqdata = SeqData(user_seqs=user_seqs, n_users=n_users, n_items=n_items)

def split_train_valid_test(seqdata: SeqData):
    train_seqs, valid_seqs, test_seqs = {}, {}, {}
    for u, seq in seqdata.user_seqs.items():
        if len(seq) < 3:
            continue
        train_seqs[u] = seq[:-2]
        valid_seqs[u] = seq[:-1]
        test_seqs[u]  = seq[:]
    return train_seqs, valid_seqs, test_seqs

train_seqs, valid_seqs, test_seqs = split_train_valid_test(seqdata)
print("Users with sequences:", len(train_seqs))

# TopPop baseline from training sequences
pop = Counter()
for seq in train_seqs.values():
    pop.update(seq)
TOPPOP = [item for item, _ in pop.most_common(2000)]
print("TopPop head:", TOPPOP[:10])

Users with sequences: 22053
TopPop head: [353, 195, 357, 387, 81, 196, 80, 331, 36, 2]


## 5) Build BM25 corpus


In [44]:
# Align docs to remapped item_id
doc_by_item_id = {}
for song_id_str, iid in item_map.items():
    doc_by_item_id[iid] = songid_to_doc.get(song_id_str, "")

def normalize_text(text: str) -> str:
    text = text.lower()
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[|,;(){}\[\]]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def expand_token(token: str) -> List[str]:
    cleaned = re.sub(r"[^\w]", "", token)
    if cleaned and cleaned != token:
        return [token, cleaned]
    return [token]

def tokenize(text: str) -> List[str]:
    text = normalize_text(text)
    tokens = []
    for tok in text.split():
        tokens.extend(expand_token(tok))
    return tokens

corpus = [tokenize(doc_by_item_id.get(i, "")) for i in range(1, n_items + 1)]
print("Example doc tokens:", corpus[0][:20])

Example doc tokens: ['faded', 'restrung', 'alan', 'walker', 'alan', 'walker', 'jesper', 'borgen', 'anders', 'froen', 'gunnar', 'greve', 'alan', 'walker', 'jesper', 'borgen', 'anders', 'froen', 'gunnar', 'greve']


In [45]:
bm25 = BM25Okapi(corpus)
print("BM25 built: corpus size =", len(corpus))

BM25 built: corpus size = 50000


## 6) Models: SASRec + SSL

- **SASRec**: Transformer encoder with causal attention
- **SSL pretraining**: CL4SRec-style InfoNCE on augmented views
- **Supervised fine-tuning**: BPR pairwise ranking loss

Evaluation uses sampled candidates

In [46]:
class SASRec(nn.Module):
    def __init__(self, n_items: int, max_len: int, d_model: int, n_heads: int, n_layers: int, dropout: float):
        super().__init__()
        self.item_emb = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.ln = nn.LayerNorm(d_model)

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        B, L = seq.shape
        pos = torch.arange(L, device=seq.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(seq) + self.pos_emb(pos)
        x = self.drop(x)
        attn_mask = torch.triu(torch.ones(L, L, device=seq.device), diagonal=1).bool()
        h = self.encoder(x, mask=attn_mask)
        return self.ln(h)

    def predict_last(self, seq: torch.Tensor) -> torch.Tensor:
        return self.forward(seq)[:, -1, :]

    def score_items(self, user_vec: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        v = self.item_emb(item_ids)
        return (v * user_vec.unsqueeze(1)).sum(-1)

def bpr_loss(pos_scores: torch.Tensor, neg_scores: torch.Tensor) -> torch.Tensor:
    return -torch.log(torch.sigmoid(pos_scores.unsqueeze(1) - neg_scores) + 1e-9).mean()

print("SASRec ready.")

SASRec ready.


In [47]:
# SSL augmentations (CL4SRec-style)
def aug_crop(seq, min_ratio=0.5):
    if len(seq) < 2:
        return seq
    L = len(seq)
    new_len = max(2, int(L * random.uniform(min_ratio, 1.0)))
    start = random.randint(0, L - new_len)
    return seq[start:start + new_len]

def aug_mask(seq, p=0.2):
    out = seq.copy()
    for i in range(len(out)):
        if random.random() < p:
            out[i] = 0
    return out

def aug_reorder(seq, window=3):
    out = seq.copy()
    if len(out) <= window:
        random.shuffle(out)
        return out
    start = random.randint(0, len(out) - window)
    sub = out[start:start + window]
    random.shuffle(sub)
    out[start:start + window] = sub
    return out

def make_view(seq):
    s = aug_crop(seq)
    s = aug_reorder(s, window=3)
    s = aug_mask(s, p=0.2)
    return s

def info_nce_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float, weights: torch.Tensor | None = None):
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    logits = (z1 @ z2.t()) / temperature
    labels = torch.arange(z1.size(0), device=z1.device)
    loss_vec = F.cross_entropy(logits, labels, reduction="none")
    if weights is not None:
        loss_vec = loss_vec * weights
        return loss_vec.sum() / (weights.sum() + 1e-9)
    return loss_vec.mean()

print("SSL components ready.")

SSL components ready.


In [48]:
def sample_negative(n_items: int, forbidden: set[int], rng: random.Random) -> int:
    while True:
        j = rng.randint(1, n_items)
        if j not in forbidden:
            return j

class SupervisedNextItemDataset(Dataset):
    def __init__(self, train_seqs: Dict[int, List[int]], max_len: int, n_items: int, n_neg: int = 10, seed: int = 42):
        self.instances = []
        rng = random.Random(seed)
        for u, seq in train_seqs.items():
            if len(seq) < 2:
                continue
            # sample a few prefixes per user
            for _ in range(min(3, len(seq) - 1)):
                t = rng.randint(1, len(seq) - 1)
                hist = seq[:t]
                pos = seq[t]
                forb = set(seq)
                negs = [sample_negative(n_items, forb, rng) for _ in range(n_neg)]
                self.instances.append((left_pad(hist, max_len), pos, negs))

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        seq, pos, negs = self.instances[idx]
        return (
            torch.tensor(seq, dtype=torch.long),
            torch.tensor(pos, dtype=torch.long),
            torch.tensor(negs, dtype=torch.long),
        )

class ContrastiveDataset(Dataset):
    def __init__(self, train_seqs: Dict[int, List[int]], max_len: int, pop_counter: Counter | None = None):
        self.users = list(train_seqs.keys())
        self.train_seqs = train_seqs
        self.max_len = max_len
        self.pop = pop_counter

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = self.users[idx]
        seq = self.train_seqs[u]
        v1 = left_pad(make_view(seq), self.max_len)
        v2 = left_pad(make_view(seq), self.max_len)
        # debias weight based on popularity of last item (simple baseline knob)
        w = 1.0
        if self.pop is not None and len(seq) > 0:
            w = 1.0 / math.sqrt(self.pop.get(seq[-1], 1))
        return torch.tensor(v1, dtype=torch.long), torch.tensor(v2, dtype=torch.long), torch.tensor(w, dtype=torch.float32)

@torch.no_grad()
def metrics_at_k(ranked: np.ndarray, target: np.ndarray, k: int):
    hit = (ranked[:, :k] == target[:, None])
    recall = hit.any(axis=1).mean()

    first_pos = hit.argmax(axis=1)
    has_hit = hit.any(axis=1)

    # MRR@K
    rr = np.zeros(hit.shape[0], dtype=np.float64)
    rr[has_hit] = 1.0 / (first_pos[has_hit] + 1.0)
    mrr = rr.mean()

    # NDCG@K (single relevant item)
    ndcg = np.zeros(hit.shape[0], dtype=np.float64)
    ndcg[has_hit] = 1.0 / np.log2(first_pos[has_hit] + 2.0)
    ndcg = ndcg.mean()

    return float(recall), float(ndcg), float(mrr)

@torch.no_grad()
def evaluate_model_sampled(
    model: SASRec,
    eval_seqs: Dict[int, List[int]],
    n_items: int,
    max_len: int,
    k: int = 20,
    M: int = 500,
    num_users: int = 5000,
    seed: int = 42,
    eval_batch_size: int = 512
):
    rng = random.Random(seed)
    users = list(eval_seqs.keys())
    rng.shuffle(users)
    users = users[:min(num_users, len(users))]

    X_all, y_all, forb_all = [], [], []
    for u in users:
        seq = eval_seqs[u]
        X_all.append(left_pad(seq[:-1], max_len))
        y_all.append(seq[-1])
        forb_all.append(set(seq))

    targets = np.array(y_all, dtype=np.int64)

    ranked_all = []

    for start in range(0, len(users), eval_batch_size):
        end = min(start + eval_batch_size, len(users))
        X = torch.tensor(X_all[start:end], dtype=torch.long, device=DEVICE)
        user_vec = model.predict_last(X)

        cand = []
        for i in range(start, end):
            forb = forb_all[i]
            c = [y_all[i]]
            while len(c) < M:
                j = rng.randint(1, n_items)
                if j not in forb:
                    c.append(j)
            cand.append(c)

        cand = torch.tensor(cand, dtype=torch.long, device=DEVICE)

        # memory-safe scoring
        scores = model.score_items(user_vec, cand)

        topk_idx = torch.topk(scores, k=k, dim=1).indices
        ranked = cand.gather(1, topk_idx).cpu().numpy()
        ranked_all.append(ranked)

        del X, user_vec, cand, scores, topk_idx
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    ranked_all = np.vstack(ranked_all)
    return metrics_at_k(ranked_all, targets, k)

@torch.no_grad()
def evaluate_toppop(eval_seqs: Dict[int, List[int]], toppop: List[int], k: int = 20, num_users: int = 20000, seed: int = 42):
    rng = random.Random(seed)
    users = list(eval_seqs.keys())
    rng.shuffle(users)
    users = users[:min(num_users, len(users))]
    ranked = np.array([toppop[:k] for _ in users], dtype=np.int64)
    target = np.array([eval_seqs[u][-1] for u in users], dtype=np.int64)
    return metrics_at_k(ranked, target, k)

print("Datasets + eval ready.")

Datasets + eval ready.


In [49]:
class EarlyStopper:
    def __init__(self, patience: int = 5, min_delta: float = 1e-4, mode: str = "max"):
        assert mode in ("max", "min")
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = None
        self.bad_epochs = 0
        self.best_state = None

    def _is_improvement(self, value: float) -> bool:
        if self.best is None:
            return True
        if self.mode == "max":
            return value > self.best + self.min_delta
        else:
            return value < self.best - self.min_delta

    def step(self, value: float, model: torch.nn.Module) -> bool:
        if self._is_improvement(value):
            self.best = value
            self.bad_epochs = 0
            # store best weights on CPU
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            return False
        else:
            self.bad_epochs += 1
            return self.bad_epochs >= self.patience

    def restore_best(self, model: torch.nn.Module):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

In [50]:
def train_sasrec_with_optional_ssl(
    n_items: int,
    train_seqs: Dict[int, List[int]],
    valid_seqs: Dict[int, List[int]],
    test_seqs: Dict[int, List[int]],
    max_len: int,
    do_ssl: bool,
    debias_info_nce: bool,
    ssl_epochs: int,
    ssl_tau: float,
    epochs: int,
    lr: float,
    batch_size: int,
    seed: int = 42,
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = SASRec(n_items=n_items, max_len=max_len, d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # 1) SSL pretraining
    if do_ssl:
        print("--- SSL pretraining (CL4SRec-style) ---")
        pop_counter = pop if debias_info_nce else None
        ds_ssl = ContrastiveDataset(train_seqs, max_len=max_len, pop_counter=pop_counter)
        dl_ssl = DataLoader(ds_ssl, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=0)

        for ep in range(ssl_epochs):
            model.train()
            total = 0.0
            for v1, v2, w in dl_ssl:
                v1 = v1.to(DEVICE)
                v2 = v2.to(DEVICE)
                w = w.to(DEVICE)

                z1 = model.predict_last(v1)
                z2 = model.predict_last(v2)

                weights = w if debias_info_nce else None
                loss = info_nce_loss(z1, z2, temperature=ssl_tau, weights=weights)

                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                total += float(loss.item())

            print(f"SSL epoch {ep+1}/{ssl_epochs}: loss={total/len(dl_ssl):.4f}")

    print("--- Supervised fine-tuning (BPR) ---")
    ds = SupervisedNextItemDataset(train_seqs, max_len=max_len, n_items=n_items, n_neg=10, seed=seed)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=0)

    stopper = EarlyStopper(patience=5, min_delta=1e-4, mode="max")

    for ep in range(epochs):
        model.train()
        total = 0.0
        for seq, pos, negs in dl:
            seq = seq.to(DEVICE)
            pos = pos.to(DEVICE)
            negs = negs.to(DEVICE)

            uvec = model.predict_last(seq)
            pos_scores = model.score_items(uvec, pos.unsqueeze(1)).squeeze(1)
            neg_scores = model.score_items(uvec, negs)

            loss = bpr_loss(pos_scores, neg_scores)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += float(loss.item())

        # validation
        r, ndcg, mrr = evaluate_model_sampled(
            model,
            valid_seqs,
            n_items=n_items,
            max_len=max_len,
            k=20,
            M=500,
            num_users=5000,
            seed=seed,
            eval_batch_size=512
        )
        print(f"Epoch {ep+1}/{epochs}: train_loss={total/len(dl):.4f} | valid R@20={r:.4f} N@20={ndcg:.4f} MRR@20={mrr:.4f}")

        # early stopping on NDCG@20
        should_stop = stopper.step(ndcg, model)
        if should_stop:
            print(f"Early stopping at epoch {ep+1}. Best valid N@20={stopper.best:.4f}")
            break

    # restore best weights before final test
    stopper.restore_best(model)

    r, ndcg, mrr = evaluate_model_sampled(
        model,
        test_seqs,
        n_items=n_items,
        max_len=max_len,
        k=20,
        M=500,
        num_users=5000,
        seed=seed,
        eval_batch_size=512
    )
    print(f"TEST (best checkpoint): R@20={r:.4f} N@20={ndcg:.4f} MRR@20={mrr:.4f}")
    return model

print("Training driver ready.")

Training driver ready.


## 7) Dense retrieval with Faiss

In [51]:
import faiss

@torch.no_grad()
def extract_item_matrix(model: SASRec) -> np.ndarray:
    W = model.item_emb.weight.detach().cpu().numpy().astype(np.float32)
    X = W[1:]
    X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    return X

def build_faiss_index(item_matrix: np.ndarray):
    D = item_matrix.shape[1]
    index = faiss.IndexFlatIP(D)  # exact IP; swap to IVF/HNSW later
    index.add(item_matrix.astype(np.float32))
    return index

@torch.no_grad()
def dense_retrieve_faiss(index, query_vec: np.ndarray, topk: int = 200):
    q = query_vec.astype(np.float32)
    q = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-12)
    scores, idx = index.search(q, topk)
    return idx + 1, scores

## 8) Sparse retrieval with BM25

In [52]:
def make_bm25_query_from_seq(seq: List[int], last_n: int = 5) -> List[str]:
    toks = []
    for iid in seq[-last_n:]:
        toks += tokenize(doc_by_item_id.get(iid, ""))
    return toks

def sparse_retrieve_bm25(bm25: BM25Okapi, query_tokens: List[str], topk: int = 200):
    scores = np.asarray(bm25.get_scores(query_tokens), dtype=np.float32)
    if topk >= scores.shape[0]:
        idx = np.argsort(scores)[::-1]
    else:
        idx = np.argpartition(scores, -topk)[-topk:]
        idx = idx[np.argsort(scores[idx])[::-1]]
    return idx + 1, scores[idx]

## 9) Fusion + reranking

In [53]:
def fuse_candidates(dense_ids: np.ndarray, sparse_ids: np.ndarray, max_cand: int = 500) -> List[int]:
    seen = set()
    fused = []
    for arr in (dense_ids, sparse_ids):
        for x in arr.tolist():
            x = int(x)
            if x not in seen:
                fused.append(x)
                seen.add(x)
            if len(fused) >= max_cand:
                return fused
    return fused

@torch.no_grad()
def rerank_by_model(model: SASRec, user_vec: torch.Tensor, cand_ids: List[int], topk: int = 20) -> np.ndarray:
    cand = torch.tensor([cand_ids], dtype=torch.long, device=user_vec.device)
    scores = model.score_items(user_vec, cand).squeeze(0)
    top_idx = torch.topk(scores, k=min(topk, scores.numel())).indices
    ranked = cand.squeeze(0).gather(0, top_idx).detach().cpu().numpy()
    return ranked

## 10) Hybrid retrieval evaluation

In [54]:
@torch.no_grad()
def eval_hybrid(
    model: SASRec,
    faiss_index,
    bm25: BM25Okapi,
    eval_seqs: Dict[int, List[int]],
    max_len: int,
    retr_k: int = 200,
    fuse_max: int = 500,
    final_k: int = 20,
    num_users: int = 5000,
    eval_batch_size: int = 256,
    seed: int = 42,
    bm25_last_n: int = 5,
):
    rng = random.Random(seed)
    users = list(eval_seqs.keys())
    rng.shuffle(users)
    users = users[:min(num_users, len(users))]

    X_all, y_all = [], []
    for u in users:
        seq = eval_seqs[u]
        X_all.append(left_pad(seq[:-1], max_len))
        y_all.append(seq[-1])
    targets = np.array(y_all, dtype=np.int64)

    ranked_dense_all = []
    ranked_sparse_all = []
    ranked_final_all = []

    t_dense = 0.0
    t_sparse = 0.0
    t_fuse_rerank = 0.0

    model.eval()

    for start in range(0, len(users), eval_batch_size):
        end = min(start + eval_batch_size, len(users))
        X = torch.tensor(X_all[start:end], dtype=torch.long, device=DEVICE)
        user_vec = model.predict_last(X)

        # Dense
        t0 = time.perf_counter()
        q = user_vec.detach().cpu().numpy().astype(np.float32)
        dense_ids, _ = dense_retrieve_faiss(faiss_index, q, topk=retr_k)
        t_dense += time.perf_counter() - t0

        # Sparse
        t0 = time.perf_counter()
        sparse_ids = []
        for i in range(start, end):
            seq = eval_seqs[users[i]]
            qtok = make_bm25_query_from_seq(seq[:-1], last_n=bm25_last_n)
            ids, _ = sparse_retrieve_bm25(bm25, qtok, topk=retr_k)
            sparse_ids.append(ids)
        sparse_ids = np.stack(sparse_ids, axis=0)
        t_sparse += time.perf_counter() - t0

        # Fuse + rerank
        t0 = time.perf_counter()
        for bi in range(end - start):
            fused = fuse_candidates(dense_ids[bi], sparse_ids[bi], max_cand=fuse_max)
            ranked = rerank_by_model(model, user_vec[bi:bi+1], fused, topk=final_k)
            ranked_final_all.append(ranked)

            ranked_dense_all.append(dense_ids[bi][:final_k])
            ranked_sparse_all.append(sparse_ids[bi][:final_k])
        t_fuse_rerank += time.perf_counter() - t0

        del X, user_vec
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    ranked_dense = np.stack(ranked_dense_all, axis=0)
    ranked_sparse = np.stack(ranked_sparse_all, axis=0)
    ranked_final = np.stack(ranked_final_all, axis=0)

    r_d, n_d, m_d = metrics_at_k(ranked_dense, targets, final_k)
    r_s, n_s, m_s = metrics_at_k(ranked_sparse, targets, final_k)
    r_f, n_f, m_f = metrics_at_k(ranked_final, targets, final_k)

    U = ranked_final.shape[0]
    latency = {
        "dense_ms": 1000.0 * t_dense / max(1, U),
        "sparse_ms": 1000.0 * t_sparse / max(1, U),
        "fuse_rerank_ms": 1000.0 * t_fuse_rerank / max(1, U),
        "total_ms": 1000.0 * (t_dense + t_sparse + t_fuse_rerank) / max(1, U),
    }

    return {"dense": (r_d, n_d, m_d),
            "sparse": (r_s, n_s, m_s),
            "hybrid": (r_f, n_f, m_f),
            "latency": latency}

## 11) Run hybrid evaluation (Supervised vs SSL)

In [55]:
#SASRec supervised only
model_sup = train_sasrec_with_optional_ssl(
    n_items=seqdata.n_items,
    train_seqs=train_seqs,
    valid_seqs=valid_seqs,
    test_seqs=test_seqs,
    max_len=MAX_SEQ_LEN,
    do_ssl=False,
    debias_info_nce=False,
    ssl_epochs=0,
    ssl_tau=SSL_TAU,
    epochs=EPOCHS,
    lr=LR,
    batch_size=BATCH_SIZE,
    seed=42,
)

#SASRec + SSL (CL4SRec-style)
model_ssl = None
if DO_SSL:
    model_ssl = train_sasrec_with_optional_ssl(
        n_items=seqdata.n_items,
        train_seqs=train_seqs,
        valid_seqs=valid_seqs,
        test_seqs=test_seqs,
        max_len=MAX_SEQ_LEN,
        do_ssl=True,
        debias_info_nce=DEBIAS_INFO_NCE,
        ssl_epochs=SSL_EPOCHS,
        ssl_tau=SSL_TAU,
        epochs=EPOCHS,
        lr=LR,
        batch_size=BATCH_SIZE,
        seed=43,
    )

--- Supervised fine-tuning (BPR) ---
Epoch 1/50: train_loss=3.8978 | valid R@20=0.1206 N@20=0.0441 MRR@20=0.0236
Epoch 2/50: train_loss=2.0680 | valid R@20=0.2260 N@20=0.0860 MRR@20=0.0478
Epoch 3/50: train_loss=1.0901 | valid R@20=0.3194 N@20=0.1446 MRR@20=0.0952
Epoch 4/50: train_loss=0.6504 | valid R@20=0.3550 N@20=0.1769 MRR@20=0.1258
Epoch 5/50: train_loss=0.4429 | valid R@20=0.3718 N@20=0.1899 MRR@20=0.1378
Epoch 6/50: train_loss=0.3385 | valid R@20=0.4002 N@20=0.2075 MRR@20=0.1522
Epoch 7/50: train_loss=0.2742 | valid R@20=0.4144 N@20=0.2187 MRR@20=0.1626
Epoch 8/50: train_loss=0.2353 | valid R@20=0.4290 N@20=0.2267 MRR@20=0.1687
Epoch 9/50: train_loss=0.2010 | valid R@20=0.4366 N@20=0.2335 MRR@20=0.1750
Epoch 10/50: train_loss=0.1793 | valid R@20=0.4438 N@20=0.2379 MRR@20=0.1780
Epoch 11/50: train_loss=0.1581 | valid R@20=0.4564 N@20=0.2466 MRR@20=0.1861
Epoch 12/50: train_loss=0.1390 | valid R@20=0.4660 N@20=0.2561 MRR@20=0.1955
Epoch 13/50: train_loss=0.1247 | valid R@20=0.47

In [56]:
# Build Faiss index for supervised model
item_mat_sup = extract_item_matrix(model_sup)
faiss_sup = build_faiss_index(item_mat_sup)

res_sup = eval_hybrid(
    model=model_sup,
    faiss_index=faiss_sup,
    bm25=bm25,
    eval_seqs=test_seqs,
    max_len=MAX_SEQ_LEN,
    retr_k=RETR_K,
    fuse_max=FUSE_MAX,
    final_k=FINAL_K,
    num_users=EVAL_USERS,
    eval_batch_size=EVAL_BATCH,
    seed=42,
)

print("--- SUPERVISED SASRec ---")
print(f"Dense  : R@{FINAL_K}={res_sup['dense'][0]:.4f} N@{FINAL_K}={res_sup['dense'][1]:.4f} MRR={res_sup['dense'][2]:.4f}")
print(f"BM25   : R@{FINAL_K}={res_sup['sparse'][0]:.4f} N@{FINAL_K}={res_sup['sparse'][1]:.4f} MRR={res_sup['sparse'][2]:.4f}")
print(f"Hybrid : R@{FINAL_K}={res_sup['hybrid'][0]:.4f} N@{FINAL_K}={res_sup['hybrid'][1]:.4f} MRR={res_sup['hybrid'][2]:.4f}")
print("Latency (ms/user):", res_sup["latency"])

if model_ssl is not None:
    item_mat_ssl = extract_item_matrix(model_ssl)
    faiss_ssl = build_faiss_index(item_mat_ssl)

    res_ssl = eval_hybrid(
        model=model_ssl,
        faiss_index=faiss_ssl,
        bm25=bm25,
        eval_seqs=test_seqs,
        max_len=MAX_SEQ_LEN,
        retr_k=RETR_K,
        fuse_max=FUSE_MAX,
        final_k=FINAL_K,
        num_users=EVAL_USERS,
        eval_batch_size=EVAL_BATCH,
        seed=43,
    )

    print("\n--- SASRec + SSL ---")
    print(f"Dense  : R@{FINAL_K}={res_ssl['dense'][0]:.4f} N@{FINAL_K}={res_ssl['dense'][1]:.4f} MRR={res_ssl['dense'][2]:.4f}")
    print(f"BM25   : R@{FINAL_K}={res_ssl['sparse'][0]:.4f} N@{FINAL_K}={res_ssl['sparse'][1]:.4f} MRR={res_ssl['sparse'][2]:.4f}")
    print(f"Hybrid : R@{FINAL_K}={res_ssl['hybrid'][0]:.4f} N@{FINAL_K}={res_ssl['hybrid'][1]:.4f} MRR={res_ssl['hybrid'][2]:.4f}")
    print("Latency (ms/user):", res_ssl["latency"])

--- SUPERVISED SASRec ---
Dense  : R@20=0.0320 N@20=0.0110 MRR=0.0054
BM25   : R@20=0.0716 N@20=0.0218 MRR=0.0085
Hybrid : R@20=0.0336 N@20=0.0111 MRR=0.0051
Latency (ms/user): {'dense_ms': 0.09378691998426802, 'sparse_ms': 282.17163417999984, 'fuse_rerank_ms': 0.26005618001217956, 'total_ms': 282.52547727999627}

--- SASRec + SSL ---
Dense  : R@20=0.0502 N@20=0.0180 MRR=0.0094
BM25   : R@20=0.0704 N@20=0.0219 MRR=0.0089
Hybrid : R@20=0.0666 N@20=0.0228 MRR=0.0112
Latency (ms/user): {'dense_ms': 0.09519305999856442, 'sparse_ms': 299.4323744199937, 'fuse_rerank_ms': 0.2557559399981983, 'total_ms': 299.7833234199905}


## Results Summary

- **SASRec (supervised):** BM25 is much better than dense retrieval in quality
  (R@20 **0.0716** vs **0.0320**), and the **hybrid method gives almost no improvement**  
  (R@20 **0.0336**, N@20 **0.0111**).

- **SASRec + SSL:** SSL clearly improves dense retrieval
  (R@20 **0.0502** vs **0.0320**) and gives the **best hybrid results**:  
  **R@20 = 0.0666**, **N@20 = 0.0228**, **MRR@20 = 0.0112**.

- **Speed:** the main bottleneck is **BM25** (about **282–299 ms/user**).  
  Dense retrieval and fuse+rerank together take **less than 0.4 ms/user**,  
  so total latency is almost fully determined by the sparse (BM25) stage.